In [ ]:
# Parameters
num_qubits = 3
target_state = '101'
iterations = 2


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import numpy as np

num_qubits = int(num_qubits)
iterations = int(iterations)
target_state = str(target_state)

if len(target_state) != num_qubits:
    target_state = '1' * num_qubits

print(f"Running Grover on {num_qubits} qubits to find |{target_state}⟩")

qc = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    qc.h(i)

for _ in range(iterations):
    qc.barrier()
    # Oracle
    for i in range(num_qubits):
        if target_state[num_qubits - 1 - i] == '0':
            qc.x(i)
    qc.h(num_qubits-1)
    qc.mcx(list(range(num_qubits-1)), num_qubits-1)
    qc.h(num_qubits-1)
    for i in range(num_qubits):
        if target_state[num_qubits - 1 - i] == '0':
            qc.x(i)
    qc.barrier()
    # Diffuser
    for i in range(num_qubits):
        qc.h(i)
        qc.x(i)
    qc.h(num_qubits-1)
    qc.mcx(list(range(num_qubits-1)), num_qubits-1)
    qc.h(num_qubits-1)
    for i in range(num_qubits):
        qc.x(i)
        qc.h(i)

qc.measure_all()

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()
print(f"Measurement results: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Statevector for Bloch sphere (qubit 0)
qc_sv = qc.remove_final_measurements(inplace=False)
sv = Statevector.from_instruction(qc_sv)
# Partial trace to get qubit 0
from qiskit.quantum_info import partial_trace
rho = partial_trace(sv, list(range(1, num_qubits)))
# Extract Bloch angles
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
